# Command line

Installing pyrosm also installs the `pyrosm` command, which you can run in a terminal. Before you download data, you can use it to check which extracts cover your study area. You can also use it to describe a PBF file that you have downloaded, to list and remove the files that pyrosm has downloaded, and to run [recipes](recipes.ipynb). The only command that downloads OpenStreetMap data is `pyrosm run`, which downloads the data that a recipe needs. By default, a recipe reuses the data that it downloaded in an earlier run. To download the data again, run the recipe with the `--force` option or set `update: true` in the recipe.

In a notebook, you can run the same commands by starting the line with `!`, as we do on this page. Let's start by listing the commands:

In [1]:
!pyrosm --help

usage: pyrosm [-h] {run,validate,find,info,downloads} ...

Run or check a pyrosm recipe, find the extracts of an area, describe a PBF
file, or list and remove the downloaded files.

positional arguments:
  {run,validate,find,info,downloads}
    run                 run a recipe and write its outputs
    validate            check a recipe without running it
    find                list the extracts of an area, best download first,
                        downloading none
    info                print a PBF file's bounding box, snapshot time and
                        SHA-256
    downloads           list or remove the files downloaded into pyrosm's
                        folder

options:
  -h, --help            show this help message and exit


As we can see, the `pyrosm` command has five subcommands, one for each of these tasks. You can get help on each of them in the same way, for example with `pyrosm find --help`. The `run` and `validate` commands are described on the [Recipes](recipes.ipynb) page, and the others below.

**Contents:**

- [Find the extracts of an area](#find-the-extracts-of-an-area)
- [Describe a PBF file](#describe-a-pbf-file)
- [List and remove downloaded files](#list-and-remove-downloaded-files)
- [Exit status](#exit-status)

## Find the extracts of an area

Before you download data, it is a good idea to check which extracts cover your study area and how large they are. The `pyrosm find` command lists the extracts that overlap an area, without downloading any of them. You can give the area in three ways:

- as a bounding box `minx,miny,maxx,maxy` in longitude and latitude, without spaces,
- as a vector file, such as a GeoPackage or GeoJSON file, whose geometries make up the area, or
- as a place name, which pyrosm turns into an area with the public Nominatim geocoding service.

For an area west of Greenwich, the longitudes are negative, so the bounding box starts with a minus sign, such as `-74.02,40.70,-73.97,40.75` in New York. You can give such a bounding box to `pyrosm find` in the same way as any other.

With the `--contains-only` option, pyrosm lists only the extracts that contain the whole area. Let's find the extracts that contain central Helsinki:

In [2]:
!pyrosm find 24.93,60.16,24.96,60.18 --contains-only

Finding the extracts took 00:09
 provider                 name      MB contains                                                              url
   BBBike             Helsinki    65.1      yes https://download.bbbike.org/osm/bbbike/Helsinki/Helsinki.osm.pbf
  Movisda    grid tile N60E024    67.2      yes      https://osm.download.movisda.io/grid/N60W024-latest.osm.pbf
  Movisda              Uusimaa   106.1      yes       https://osm.download.movisda.io/admin/FI-18-latest.osm.pbf
  Movisda              Finland   715.3      yes          https://osm.download.movisda.io/admin/FI-latest.osm.pbf
Geofabrik              Finland   774.3      yes      https://download.geofabrik.de/europe/finland-latest.osm.pbf
  Movisda grid tile N60E020-10   906.7      yes   https://osm.download.movisda.io/grid/N60W020-10-latest.osm.pbf
Geofabrik               Europe 35342.7      yes              https://download.geofabrik.de/europe-latest.osm.pbf


The table shows the provider of each extract, its name, its size in megabytes, whether it contains the area, and its web address. Because we used `--contains-only`, every extract in the table contains the area. Pyrosm orders the extracts by file size, with the smallest first. Extracts whose size is unknown come last. The `get_data_by_area()` function uses the same order and, by default, downloads the first extract that contains the area. In our case, it is BBBike's extract of Helsinki (65 MB). Without `--contains-only`, pyrosm first lists the extracts that contain the area. The extracts that cover only part of the area come after them, again ordered by file size.

Notice that in the web addresses of Movisda's grid tiles, the letters E and W are swapped: the file of the tile at 24 degrees east is called `N60W024`. This is how Movisda names its files. Pyrosm names the tiles by their position, so the name `N60E024` in the table is correct.

To find the extracts, pyrosm downloads the list of extracts that each provider publishes and keeps these lists in its default folder. Pyrosm reuses the lists for a day. After that, it checks whether the providers have updated them. If pyrosm cannot download a list, it shows a warning and uses a copy that it already has. This can be a copy that it downloaded earlier or one that comes with pyrosm. Pyrosm does not come with a copy of Movisda's list of administrative areas. If pyrosm cannot download this list and has no copy from an earlier download, it leaves Movisda's administrative extracts out of the results.

With the `--json` option, `pyrosm find` prints the extracts as JSON, which is easier to process in a script. In the JSON output, each extract has the fields `provider`, `id`, `name`, `url`, `bytes`, `contains`, and `bounds`. The `bytes` field gives the size of the file, or `null` if the size is unknown.

## Describe a PBF file

The `pyrosm info` command prints the main facts of a PBF file: its size in bytes, its bounding box, the time of its snapshot, and its SHA-256 checksum. If the file does not record its bounding box or snapshot time, pyrosm prints `none` instead. The snapshot time tells when the data was taken from OpenStreetMap (in UTC), as written in the header of the file. The checksum is a fingerprint of the file's content. You can use it, for example, to make sure that a recipe always uses exactly the same data (see [Pin the data](recipes.ipynb#pin-the-data)).

Let's download Geofabrik's extract of Monaco with `get_data()` and describe it. In a notebook, `{fp}` inserts the value of the Python variable `fp` into the command:

In [3]:
import pyrosm

fp = pyrosm.get_data("monaco")

In [4]:
!pyrosm info {fp}

file: /tmp/pyrosm/monaco-latest.osm.pbf
bytes: 706868
bounding box: 7.408583, 43.483817, 7.595671, 43.75293
snapshot: 2026-10-30T21:24:51+00:00
sha256: 5b8382ba1ded2bd7f89573c3fcdfbef4aabccbe8165bb7cf53300d4860cdebca


As we can see, the file is 706,736 bytes (0.7 MB) in size, and its data was taken from OpenStreetMap on 28 October 2026 at 21:21 UTC. The bounding box is in the same order as in `pyrosm find`: minx, miny, maxx, maxy. With the `--json` option, `pyrosm info` prints the same information as JSON.

## List and remove downloaded files

By default, pyrosm downloads files into a folder called `pyrosm` in the temporary folder of your computer. You can choose another folder with the `directory` parameter of `get_data()` and the other downloading functions. The `pyrosm downloads list` command prints the path of each PBF file in pyrosm's default folder:

In [5]:
!pyrosm downloads list

/tmp/pyrosm/monaco-latest.osm.pbf


The `pyrosm downloads clear` command removes these files. Without a file name, it removes every PBF file in pyrosm's default folder. If you give it the name of a file, it removes only that file:

In [6]:
!pyrosm downloads clear monaco-latest.osm.pbf

removed 1 file


The name of the file needs to end with `.osm.pbf` or `.osh.pbf`. When pyrosm removes a PBF file, it also removes the incomplete copies that interrupted downloads have left behind. It also removes the small files that it keeps next to the PBF file to record how the file was downloaded or cropped. When it clears the default folder, pyrosm leaves the providers' lists of extracts in place. The command leaves the layer cache in place as well. This cache contains the layers that pyrosm has read before. The files that are installed with pyrosm, such as the Helsinki test data, are in another folder, so the command leaves them in place too.

Without a path, the command does not touch the files in a folder that you chose yourself with `directory`. To remove one of these files, give its path to `pyrosm downloads clear`.

Notice that once you remove a file, pyrosm has to download it again the next time you need it. Remove a file only when you no longer need it (see [Download responsibly](download_responsibly.ipynb)).

## Exit status

When a `pyrosm` command finishes, it returns an **exit status**, a number that scripts and workflow tools can check. A status of 0 means that the command succeeded. A status of 1 means that pyrosm understood the command but stopped because of a problem while carrying it out, for example because a file does not exist, the data cannot be read, or a download fails. In this case, pyrosm prints a message that explains the problem. A status of 2 means that the command itself was written incorrectly, for example with an unknown option. Let's ask pyrosm to describe a file that does not exist, and then print the exit status with `echo $?`:

In [7]:
!pyrosm info missing.osm.pbf; echo "exit status: $?"

pyrosm: [Errno 2] No such file or directory: 'missing.osm.pbf'
exit status: 1


The commands never ask for input while they run. This means that you can use them in shell scripts and in workflow tools, such as Snakemake (see [Use recipes in scripts and workflows](recipes.ipynb#use-recipes-in-scripts-and-workflows)).